In [1]:
import pandas as pd 
multi_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT/TACRED_full_binary_question_result.csv')

In [2]:
multi_df.columns[3:]

Index(['subject', 'subject_type', 'object', 'object_type', 'subj_start',
       'subj_end', 'obj_start', 'obj_end', 'docid', 'token',
       ...
       'profession', 'holds_advisory_position_for', 'occupation-title',
       'is_a_creator', 'Title-Affiliation', 'author_of',
       'expresses_affection_to', 'title or position held', 'has_profession',
       'title or honorific of person'],
      dtype='object', length=951)

In [3]:
multi_df.head(2)

,id,relation,text,subject,subject_type,object,object_type,subj_start,subj_end,obj_start,...,profession,holds_advisory_position_for,occupation-title,is_a_creator,Title-Affiliation,author_of,expresses_affection_to,title or position held,has_profession,title or honorific of person
0,61b3aa9e36d5d8427e8e,org:alternate_names,"DC : Nov 30th thru Dec 2nd , Servicemembers Un...",Servicemembers United,ORGANIZATION,Servicemembers Legal Defense Network,ORGANIZATION,8,9,28,...,0,0,0,0,0,0,0,0,0,0
1,61b3aa9e368a0e3dfbd2,org:alternate_names,With the tally completed in more than half the...,All Basotho Convention,ORGANIZATION,ABC,ORGANIZATION,29,31,33,...,0,0,0,0,1,0,0,1,0,0


In [6]:
import ast 

multi_df = multi_df.rename(columns={'gpt4_phrase': 'predicted_relation'})

multi_df['multi_labels'] = multi_df['multi_labels'].apply(ast.literal_eval) 
labels_series = multi_df['multi_labels'].explode() 
labels_count = labels_series.value_counts().reset_index() 
labels_count.columns = ['predicted_relation', 'num_samples'] 

relation_to_ids = ( 
    multi_df
        .explode('multi_labels')                        
        .groupby('multi_labels')['id']                  
        .apply(list)                                    
        .to_dict()                                      
)


labels_count['id_list'] = labels_count['predicted_relation'].map(relation_to_ids) 

relations = labels_count['predicted_relation'].tolist() 


for rel in relations: 
    labels_count[rel] = 0
    

id_sets = [set(ids) for ids in labels_count['id_list']]


for i, rel_i in enumerate(relations): 
    ids_i = id_sets[i] 
    for j, rel_j in enumerate(relations): 
        ids_j = id_sets[j] 
        overlap = len(ids_i & ids_j)   
        labels_count.at[i, rel_j] = overlap
labels_count.head(2)

/tmp/ipykernel_1006855/3254683051.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labels_count[rel] = 0
/tmp/ipykernel_1006855/3254683051.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labels_count[rel] = 0
/tmp/ipykernel_1006855/3254683051.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

,predicted_relation,num_samples,id_list,association_with,ownership or affiliation,associate_of,affiliated_with,Union_Location_Association,Reference_Link,Composition-Membership,...,credit_rating_agency_action,collaborative source of gas price information,Pioneer_In_American_Judiciary,co-founders of fashion line,Convicted_For_Supporting_Rebels,served_as_press_aide_to,collaborator_in_mountain_ascent,fear_of_extradition_to_location,film_release_date,appointed_as_chancellor_of_public_school_system_in
0,association_with,1601,"[61b3aa9e36d5d8427e8e, 61b3aa9e368a0e3dfbd2, e...",1601,1202,938,943,797,770,763,...,1,1,1,1,1,1,1,0,0,1
1,ownership or affiliation,1239,"[61b3aa9e36d5d8427e8e, 61b3aa9e368a0e3dfbd2, e...",1202,1239,862,897,660,640,720,...,0,1,1,1,1,1,1,0,0,1


In [ ]:
labels_count.to_csv('/workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT/TACRED_full_intersection_matrix.csv', index=False)